In [ ]:
!pip install -q imbalanced-learn xgboost lightgbm catboost gradio

# **target**
# **الهدف: التنبؤ بما إذا كان العميل سيشترك في وديعة بنكية لأجل (`deposit`: yes = 1، no = 0) بناءً على بيانات حملة تسويق بنكي.**

**(col 1) age:** عمر العميل.

**(col 2) job:** نوع الوظيفة (admin., technician, management, retired, student, ...).

**(col 3) marital:** الحالة الاجتماعية (married, single, divorced).

**(col 4) education:** المستوى التعليمي (primary, secondary, tertiary, unknown).

**(col 5) default:** هل عنده ائتمان متعثر (yes/no).

**(col 6) balance:** متوسط الرصيد السنوي بالـ Euro.

**(col 7) housing:** هل عنده قرض سكني (yes/no).

**(col 8) loan:** هل عنده قرض شخصي (yes/no).

**(col 9) contact:** وسيلة التواصل (cellular, telephone, unknown).

**(col 10) day:** يوم آخر اتصال في الشهر.

**(col 11) month:** شهر آخر اتصال في السنة.

**(col 12) duration:** مدة آخر مكالمة بالثواني. ⚠️ بتتعرف **بعد** المكالمة، فلو الهدف توقع قبل الاتصال لازم تتشال (هنقارن في آخر النوت بوك).

**(col 13) campaign:** عدد مرات الاتصال بالعميل في الحملة الحالية.

**(col 14) pdays:** عدد الأيام من آخر اتصال في حملة سابقة (`-1` = ما اتصلش بيه قبل كده).

**(col 15) previous:** عدد مرات الاتصال بالعميل قبل الحملة دي.

**(col 16) poutcome:** نتيجة الحملة السابقة (success, failure, other, unknown).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')

In [ ]:
RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('/content/bank.csv')

# **EDA**

In [ ]:
display(df.head())

In [ ]:
df.info()

In [ ]:
print('Shape:', df.shape, '| Duplicates:', df.duplicated().sum())

In [ ]:
display(df.describe().T)

In [ ]:
missing = df.isna().sum().to_frame('missing')

In [ ]:
missing['%'] = (missing['missing'] / len(df) * 100).round(1)
display(missing[missing['missing'] > 0])

**فيه قيم ناقصة في أغلب الأعمدة (بنسب صغيرة)، وفي 9 صفوف الـ target نفسه ناقص، فهنشيلهم لأن مفيش طريقة سليمة نعوّض بيها الـ target.**

In [ ]:
df = df.dropna(subset=['deposit']).reset_index(drop=True)
df['deposit'] = df['deposit'].map({'no': 0, 'yes': 1}).astype(int)

class_share = df['deposit'].value_counts(normalize=True).round(3)

In [ ]:
print('Class share:\n', class_share)

In [ ]:
print(f"\n Baseline accuracy (توقع الفئة الأكبر دايماً): {class_share.max():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df, x='deposit', ax=axes[0]).set_title('Target Distribution')
corr = df.select_dtypes('number').corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=.5, ax=axes[1], cbar=False)
axes[1].set_title('Correlation')
plt.tight_layout()
plt.show()

In [ ]:
num_features = df.drop(columns='deposit').select_dtypes('number').columns
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_features):
    sns.histplot(data=df, x=col, hue='deposit', kde=True, bins=30, ax=ax, element='step')
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_features):
    sns.boxplot(data=df, x='deposit', y=col, ax=ax)
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
cat_features = df.drop(columns='deposit').select_dtypes(exclude='number').columns
fig, axes = plt.subplots(3, 3, figsize=(18, 13))
for ax, col in zip(axes.ravel(), cat_features):
    rate = df.groupby(col)['deposit'].mean().sort_values()
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color='#1f77b4')
    ax.set_title(f'Deposit rate by {col}'); ax.set_xlabel('Share of deposit = yes')
plt.tight_layout(); plt.show()

**الـ `duration` أقوى عمود في الارتباط بالـ target، لكنه بيتعرف بعد المكالمة (تسريب لو بنتنبأ قبلها).**
**الـ outliers في `balance` و`campaign` و`previous` طبيعية في بيانات البنوك فمش هنمسحها.**

# Data Split

In [ ]:
x = df.drop('deposit', axis=1)
y = df['deposit']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(x_train.shape, x_test.shape)

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE

def build_preprocessor(frame):
    num_cols = frame.select_dtypes('number').columns.tolist()
    cat_cols = frame.select_dtypes(exclude='number').columns.tolist()
    return ColumnTransformer([
        ('num', ImbPipeline([('imputer', SimpleImputer(strategy='median')),
                             ('scaler', StandardScaler())]), num_cols),
        ('cat', ImbPipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                             ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
    ])

def make_pipe(model, frame=None):
    frame = x_train if frame is None else frame
    return ImbPipeline([
        ('prep', build_preprocessor(frame)),
        ('model', model),
    ])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_validate

In [ ]:
base_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'KNN': KNeighborsClassifier(n_neighbors=15),
    'SVC': SVC(probability=True, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(max_depth=3, learning_rate=0.05, n_estimators=150, random_state=RANDOM_STATE),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=4, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE),
    'LightGBM': LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=15, random_state=RANDOM_STATE, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=300, depth=5, learning_rate=0.05, verbose=0, random_state=RANDOM_STATE),
}

rows = {}
for name, m in base_models.items():
    cv = cross_validate(make_pipe(m), x_train, y_train, cv=skf,
                        scoring=['accuracy', 'balanced_accuracy', 'f1', 'roc_auc'])
    rows[name] = {'CV Accuracy': cv['test_accuracy'].mean(),
                  'CV Balanced Acc': cv['test_balanced_accuracy'].mean(),
                  'CV F1': cv['test_f1'].mean(),
                  'CV ROC-AUC': cv['test_roc_auc'].mean()}

cv_table = pd.DataFrame(rows).T.sort_values('CV Accuracy', ascending=False).round(4)
cv_table

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
search_spaces = {
    'SVC': (
        SVC(probability=False, random_state=RANDOM_STATE),
        {'model__C': loguniform(0.3, 50), 'model__gamma': loguniform(0.005, 0.5)}, 15),
    'Extra Trees': (
        ExtraTreesClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
        {'model__max_depth': [None, 12, 20, 30], 'model__min_samples_leaf': randint(1, 6),
         'model__max_features': ['sqrt', 'log2', 0.5]}, 15),
    'Random Forest': (
        RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
        {'model__max_depth': [None, 8, 12, 16, 24], 'model__min_samples_leaf': randint(1, 8),
         'model__max_features': ['sqrt', 'log2', 0.5]}, 15),
    'XGBoost': (
        XGBClassifier(random_state=RANDOM_STATE),
        {'model__n_estimators': randint(100, 500), 'model__learning_rate': loguniform(0.01, 0.2),
         'model__max_depth': randint(2, 7), 'model__subsample': uniform(0.6, 0.4),
         'model__colsample_bytree': uniform(0.6, 0.4), 'model__min_child_weight': randint(1, 10)}, 20),
    'LightGBM': (
        LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
        {'model__n_estimators': randint(100, 500), 'model__learning_rate': loguniform(0.01, 0.2),
         'model__num_leaves': randint(4, 40), 'model__min_child_samples': randint(10, 60),
         'model__subsample': uniform(0.6, 0.4), 'model__colsample_bytree': uniform(0.6, 0.4)}, 20),
}

tuned = {}
for name, (model, grid, n_iter) in search_spaces.items():
    search = RandomizedSearchCV(make_pipe(model), grid, n_iter=n_iter, cv=skf, scoring='accuracy',
                                random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(x_train, y_train)
    best = search.best_estimator_
    if name == 'SVC':
        # الـ probability بتبطّأ الـ search، فبنشغّلها بعد اختيار أحسن parameters (محتاجينها للـ Voting/Stacking)
        best.set_params(model__probability=True).fit(x_train, y_train)
    tuned[name] = best
    print(f"{name:14s} CV accuracy = {search.best_score_:.4f}")

# **Ensembles: Voting & Stacking (على الموديلات المضبوطة)**
# **دمج أفضل النماذج المضبوطة معاً باستخدام VotingClassifier (Soft Voting) و StackingClassifier لتقليل الأخطاء الفردية ورفع استقرار التوقع.**

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier

In [ ]:
estimators = [(n.replace(' ', '_'), m) for n, m in tuned.items()]

voting = VotingClassifier(estimators=estimators, voting='soft')
stacking = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(max_iter=2000),
                              cv=5, stack_method='predict_proba')

voting.fit(x_train, y_train)
stacking.fit(x_train, y_train)

final_models = {**tuned, 'Voting (soft)': voting, 'Stacking': stacking}

# **Final Evaluation على الـ Test Set**

# **اختبار النماذج نهائياً على بيانات الاختبار واستخراج (Accuracy, Balanced Accuracy, F1, ROC-AUC) مع رسم الـ Confusion Matrix و ROC Curves.**

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

In [ ]:
def evaluate(model):
    pred = model.predict(x_test)
    prob = model.predict_proba(x_test)[:, 1]
    return {
        'Accuracy': accuracy_score(y_test, pred),
        'Balanced Acc': balanced_accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob),
    }

results = pd.DataFrame({n: evaluate(m) for n, m in final_models.items()}).T
results = results.sort_values('Accuracy', ascending=False).round(4)
baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Baseline (توقع الفئة الأكبر دايماً): {baseline:.2f}")
results

In [ ]:
results[['Accuracy', 'Balanced Acc', 'F1', 'ROC-AUC']].plot.bar(figsize=(12, 5))
plt.axhline(baseline, color='k', linestyle='--', label='Baseline accuracy')
plt.ylim(0.4, 1.0); plt.legend(loc='lower right'); plt.title('Test Metrics'); plt.tight_layout(); plt.show()

In [ ]:
best_name = results.index[0]
best_model = final_models[best_name]
pred = best_model.predict(x_test)
print(f'Best model: {best_name}\n')
print(classification_report(y_test, pred, target_names=['No Deposit (0)', 'Deposit (1)']))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['No Deposit', 'Deposit'], yticklabels=['No Deposit', 'Deposit'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual'); axes[0].set_title(f'Confusion Matrix - {best_name}')
for n, m in final_models.items():
    prob = m.predict_proba(x_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[1].plot(fpr, tpr, label=f"{n} ({roc_auc_score(y_test, prob):.2f})")
axes[1].plot([0, 1], [0, 1], 'k--'); axes[1].set_title('ROC Curves'); axes[1].legend(fontsize=8)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
plt.tight_layout(); plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from imblearn.pipeline import Pipeline as ImbPipeline

core = clone(tuned['Extra Trees'].named_steps['model'])
with_smote = ImbPipeline([
    ('prep', build_preprocessor(x_train)),
    ('smote', SMOTE(random_state=RANDOM_STATE)), ('model', core)])
without_smote = make_pipe(clone(core))

cmp = {}
for label, p in [('Without SMOTE', without_smote), ('With SMOTE', with_smote)]:
    p.fit(x_train, y_train)
    pred = p.predict(x_test)
    cmp[label] = {'Accuracy': accuracy_score(y_test, pred), 'Recall (Deposit)': recall_score(y_test, pred),
                  'Balanced Acc': balanced_accuracy_score(y_test, pred), 'F1': f1_score(y_test, pred)}
pd.DataFrame(cmp).T.round(4)

**الـ classes هنا متوازنة تقريباً (52% / 48%)، فالـ SMOTE مش متوقع يفيد، والمقارنة دي للتأكيد.**

# **Feature Importance**

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
perm = permutation_importance(best_model, x_test, y_test, n_repeats=5, random_state=RANDOM_STATE,
                              scoring='accuracy', n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=x.columns).sort_values()
imp.plot.barh(figsize=(8, 6), xerr=pd.Series(perm.importances_std, index=x.columns)[imp.index])
plt.title(f'Permutation Importance - {best_name}'); plt.xlabel('Drop in accuracy'); plt.tight_layout(); plt.show()

# **قبل وبعد حذف duration**
**الـ `duration` بيتعرف بعد المكالمة، فمينفعش يتستخدم لو الهدف توقع قبل الاتصال. هنقارن نفس الموديل (LightGBM المضبوط) بيه وبدونه.**

In [ ]:
x_train_nd = x_train.drop(columns='duration')
x_test_nd = x_test.drop(columns='duration')

lgbm_core = tuned['LightGBM'].named_steps['model']
no_duration = make_pipe(clone(lgbm_core), x_train_nd).fit(x_train_nd, y_train)

def metrics(model, X):
    pred, prob = model.predict(X), model.predict_proba(X)[:, 1]
    return {'Accuracy': accuracy_score(y_test, pred), 'Balanced Acc': balanced_accuracy_score(y_test, pred),
            'F1': f1_score(y_test, pred), 'ROC-AUC': roc_auc_score(y_test, prob)}

pd.DataFrame({'With duration': metrics(tuned['LightGBM'], x_test),
              'Without duration': metrics(no_duration, x_test_nd)}).T.round(4)

# **Save Model + Predict**

In [ ]:
import joblib

# حفظ أفضل موديل (الـ Pipeline أو الـ Voting/Stacking المجمع) في ملف pkl
joblib.dump(best_model, 'bank_deposit_pipeline.pkl')
print("✅ تم حفظ الموديل بنجاح باسم: bank_deposit_pipeline.pkl")

loaded = joblib.load('bank_deposit_pipeline.pkl')
sample = x_test.iloc[[0]]          # الـ Pipeline بيتعامل مع القيم الناقصة والـ encoding لوحده
print('Prediction:', 'Deposit' if loaded.predict(sample)[0] == 1 else 'No Deposit',
      '| Prob(Deposit) =', round(loaded.predict_proba(sample)[0, 1], 3))

In [ ]:
import gradio as gr

model = joblib.load('bank_deposit_pipeline.pkl')
FEATURES = x.columns.tolist()

def predict_deposit(*values):
    row = [np.nan if (v is None or v == '') else v for v in values]
    input_data = pd.DataFrame([row], columns=FEATURES)
    for c in x_train.select_dtypes('number').columns:
        input_data[c] = pd.to_numeric(input_data[c], errors='coerce')
    prediction = int(model.predict(input_data)[0])
    prob = float(model.predict_proba(input_data)[0][1])
    if prediction == 1:
        return f"✅ هيشترك في الوديعة (Deposit) - احتمال: {prob * 100:.1f}%"
    return f"🚫 مش هيشترك (No Deposit) - احتمال: {(1 - prob) * 100:.1f}%"

inputs = []
for c in FEATURES:
    if c in x_train.select_dtypes('number').columns:
        inputs.append(gr.Number(label=c, value=float(x_train[c].median())))
    else:
        inputs.append(gr.Dropdown(choices=sorted(x_train[c].dropna().unique().tolist()), value=x_train[c].mode()[0], label=c))

# أمثلة جاهزة: صفوف حقيقية من الـ test set (3 اشتركوا فعلاً + 3 ما اشتركوش)، تتجرّب بضغطة واحدة
pos = x_test[y_test == 1].sample(3, random_state=RANDOM_STATE)
neg = x_test[y_test == 0].sample(3, random_state=RANDOM_STATE)
examples_df = pd.concat([pos.iloc[[0]], neg.iloc[[0]], pos.iloc[[1]], neg.iloc[[1]], pos.iloc[[2]], neg.iloc[[2]]])
num_cols = x_train.select_dtypes('number').columns

examples = [
    [None if pd.isna(v) else (float(v) if c in num_cols else v) for c, v in row.items()]
    for _, row in examples_df.iterrows()
]
print(pd.DataFrame({'Example': range(1, len(examples) + 1),
                    'Actual deposit': y_test.loc[examples_df.index].map({0: 'No', 1: 'Yes'}).values}))

demo = gr.Interface(
    fn=predict_deposit,
    inputs=inputs,
    outputs=gr.Textbox(label="نتيجة التقييم"),
    title="🏦 Bank Term Deposit Prediction",
    description="أدخل بيانات العميل لمعرفة احتمال اشتراكه في وديعة بنكية، أو اضغط على أي مثال من تحت (صفوف حقيقية من الـ test set).",
    examples=examples,
    cache_examples=False,
)

if __name__ == "__main__":
    demo.launch()